In [187]:
# January 2025 Data Cleaning

from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
PJM_PRICE_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "pjm"
    / "da_hrl_lmps_PJM_PS.csv"
)

PJM_LOAD_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "pjm"
    / "hrl_load_metered_PJM_PS.csv"
)

NYISO_PRICE_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "nyiso"
    / "nyiso_hudson_valley_jan2025_LMP_DATA.csv"
)

NYISO_LOAD_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "nyiso"
    / "nyiso_hudson_valley_jan2025palIntegrated_HV_loaddata.csv"
)


NOAA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "noaa"
)

NEWARK_WEATHER_FILE = (
    NOAA_DIR
    / "WeatherData Jan25 Newark.csv"
)

STEWART_WEATHER_FILE = (
    NOAA_DIR
    / "WeatherData Jan25 Stewart.csv"
)

newark_weather_raw = pd.read_csv(
    NEWARK_WEATHER_FILE,
    low_memory=False,
)

stewart_weather_raw = pd.read_csv(
    STEWART_WEATHER_FILE,
    low_memory=False,
)

pjm_price_raw = pd.read_csv(PJM_PRICE_FILE)
pjm_load_raw = pd.read_csv(PJM_LOAD_FILE)
nyiso_price_raw =pd.read_csv(NYISO_PRICE_FILE)
nyiso_load_raw = pd.read_csv(NYISO_LOAD_FILE)


print("PJM price:", pjm_price_raw.shape)
print("PJM load:", pjm_load_raw.shape)
print("NYISO price:", nyiso_price_raw.shape)
print("NYISO load:", nyiso_load_raw.shape)
print("Newark file exists:", NEWARK_WEATHER_FILE.exists())
print("Stewart file exists:", STEWART_WEATHER_FILE.exists())

print("Newark weather shape:", newark_weather_raw.shape)
print("Stewart weather shape:", stewart_weather_raw.shape)

print(nyiso_price_raw.columns.tolist())
print(nyiso_load_raw.columns.tolist())

PJM price: (744, 14)
PJM load: (744, 8)
NYISO price: (744, 7)
NYISO load: (744, 6)
Newark file exists: True
Stewart file exists: True
Newark weather shape: (12970, 125)
Stewart weather shape: (9081, 125)
['Time Stamp', 'Name', 'PTID', 'LBMP ($/MWHr)', 'Marginal Cost Losses ($/MWHr)', 'Marginal Cost Congestion ($/MWHr)', 'source_file']
['Time Stamp', 'Time Zone', 'Name', 'PTID', 'Integrated Load', 'source_file']


In [188]:
## Load raw data
print(pjm_price_raw.shape)
pjm_price_raw.head()

(744, 14)


,datetime_beginning_utc,datetime_beginning_ept,pnode_id,pnode_name,voltage,equipment,type,zone,system_energy_price_da,total_lmp_da,congestion_price_da,marginal_loss_price_da,row_is_current,version_nbr
0,1/1/2025 5:00:00 AM,1/1/2025 12:00:00 AM,51301,PSEG,NaN,NaN,ZONE,NaN,21.26,21.981200,0.153437,0.567763,True,1
1,1/1/2025 6:00:00 AM,1/1/2025 1:00:00 AM,51301,PSEG,NaN,NaN,ZONE,NaN,20.96,21.397873,-0.012770,0.450643,True,1
2,1/1/2025 7:00:00 AM,1/1/2025 2:00:00 AM,51301,PSEG,NaN,NaN,ZONE,NaN,20.42,20.641678,-0.266470,0.488148,True,1
3,1/1/2025 8:00:00 AM,1/1/2025 3:00:00 AM,51301,PSEG,NaN,NaN,ZONE,NaN,20.45,20.696320,-0.283920,0.530240,True,1
4,1/1/2025 9:00:00 AM,1/1/2025 4:00:00 AM,51301,PSEG,NaN,NaN,ZONE,NaN,20.46,20.649481,-0.400733,0.590214,True,1


In [189]:
pjm_price = pjm_price_raw.copy()

pjm_price["timestamp_local"] = pd.to_datetime(
    pjm_price["datetime_beginning_ept"],
    format="%m/%d/%Y %I:%M:%S %p",
)

pjm_price["timestamp_utc"] = pd.to_datetime(
    pjm_price["datetime_beginning_utc"],
    format="%m/%d/%Y %I:%M:%S %p",
    utc=True,
)

In [190]:
pjm_price = pjm_price[
    [
        "timestamp_local",
        "timestamp_utc",
        "pnode_id",
        "pnode_name",
        "total_lmp_da",
        "system_energy_price_da",
        "congestion_price_da",
        "marginal_loss_price_da",
    ]
].rename(
    columns={
        "pnode_id": "location_id",
        "pnode_name": "location",
        "total_lmp_da": "day_ahead_price_usd_mwh",
        "system_energy_price_da": "energy_component_usd_mwh",
        "congestion_price_da": "congestion_component_usd_mwh",
        "marginal_loss_price_da": "loss_component_usd_mwh",
    }
)

pjm_price = pjm_price.sort_values(
    "timestamp_local"
).reset_index(drop=True)

In [191]:
print("Shape:", pjm_price.shape)
print("First local hour:", pjm_price["timestamp_local"].min())
print("Last local hour:", pjm_price["timestamp_local"].max())
print(
    "Duplicate local hours:",
    pjm_price["timestamp_local"].duplicated().sum(),
)
print(
    "Duplicate UTC hours:",
    pjm_price["timestamp_utc"].duplicated().sum(),
)
print(
    "Missing prices:",
    pjm_price["day_ahead_price_usd_mwh"].isna().sum(),
)

pjm_price.head()

Shape: (744, 8)
First local hour: 2025-01-01 00:00:00
Last local hour: 2025-01-31 23:00:00
Duplicate local hours: 0
Duplicate UTC hours: 0
Missing prices: 0


,timestamp_local,timestamp_utc,location_id,location,day_ahead_price_usd_mwh,energy_component_usd_mwh,congestion_component_usd_mwh,loss_component_usd_mwh
0,2025-01-01 00:00:00,2025-01-01 05:00:00+00:00,51301,PSEG,21.981200,21.26,0.153437,0.567763
1,2025-01-01 01:00:00,2025-01-01 06:00:00+00:00,51301,PSEG,21.397873,20.96,-0.012770,0.450643
2,2025-01-01 02:00:00,2025-01-01 07:00:00+00:00,51301,PSEG,20.641678,20.42,-0.266470,0.488148
3,2025-01-01 03:00:00,2025-01-01 08:00:00+00:00,51301,PSEG,20.696320,20.45,-0.283920,0.530240
4,2025-01-01 04:00:00,2025-01-01 09:00:00+00:00,51301,PSEG,20.649481,20.46,-0.400733,0.590214


In [192]:
pjm_load = pjm_load_raw.copy()

pjm_load["timestamp_local"] = pd.to_datetime(
    pjm_load["datetime_beginning_ept"],
    format="%m/%d/%Y %I:%M:%S %p",
)

pjm_load["timestamp_utc"] = pd.to_datetime(
    pjm_load["datetime_beginning_utc"],
    format="%m/%d/%Y %I:%M:%S %p",
    utc=True,
)

pjm_load = pjm_load[
    [
        "timestamp_local",
        "timestamp_utc",
        "zone",
        "load_area",
        "mw",
        "is_verified",
    ]
].rename(
    columns={
        "mw": "actual_load_mw",
    }
)

pjm_load = pjm_load.sort_values(
    "timestamp_local"
).reset_index(drop=True)

In [193]:
print("Shape:", pjm_load.shape)
print("First local hour:", pjm_load["timestamp_local"].min())
print("Last local hour:", pjm_load["timestamp_local"].max())
print(
    "Duplicate local hours:",
    pjm_load["timestamp_local"].duplicated().sum(),
)
print(
    "Missing load values:",
    pjm_load["actual_load_mw"].isna().sum(),
)
print(
    "Unverified observations:",
    (~pjm_load["is_verified"]).sum(),
)

assert len(pjm_load) == 744
assert pjm_load["timestamp_local"].is_unique
assert pjm_load["timestamp_utc"].is_unique
assert pjm_load["actual_load_mw"].notna().all()
assert pjm_load["is_verified"].all()

pjm_load.head()

Shape: (744, 6)
First local hour: 2025-01-01 00:00:00
Last local hour: 2025-01-31 23:00:00
Duplicate local hours: 0
Missing load values: 0
Unverified observations: 0


,timestamp_local,timestamp_utc,zone,load_area,actual_load_mw,is_verified
0,2025-01-01 00:00:00,2025-01-01 05:00:00+00:00,PS,PS,4068.850,True
1,2025-01-01 01:00:00,2025-01-01 06:00:00+00:00,PS,PS,3966.606,True
2,2025-01-01 02:00:00,2025-01-01 07:00:00+00:00,PS,PS,3851.354,True
3,2025-01-01 03:00:00,2025-01-01 08:00:00+00:00,PS,PS,3767.111,True
4,2025-01-01 04:00:00,2025-01-01 09:00:00+00:00,PS,PS,3766.670,True


In [194]:
nyiso_price = nyiso_price_raw.copy()

nyiso_price["timestamp_local"] = pd.to_datetime(
    nyiso_price["Time Stamp"],
    format="%Y-%m-%d %H:%M:%S",
)

nyiso_price["timestamp_utc"] = (
    nyiso_price["timestamp_local"]
    .dt.tz_localize("America/New_York")
    .dt.tz_convert("UTC")
)

nyiso_price = nyiso_price[
    [
        "timestamp_local",
        "timestamp_utc",
        "PTID",
        "Name",
        "LBMP ($/MWHr)",
        "Marginal Cost Losses ($/MWHr)",
        "Marginal Cost Congestion ($/MWHr)",
    ]
].rename(
    columns={
        "PTID": "location_id",
        "Name": "location",
        "LBMP ($/MWHr)": "day_ahead_price_usd_mwh",
        "Marginal Cost Losses ($/MWHr)":
            "loss_component_usd_mwh",
        "Marginal Cost Congestion ($/MWHr)":
            "congestion_component_usd_mwh",
    }
)

nyiso_price = nyiso_price.sort_values(
    "timestamp_local"
).reset_index(drop=True)

In [195]:
nyiso_price["energy_component_usd_mwh"] = (
    nyiso_price["day_ahead_price_usd_mwh"]
    - nyiso_price["loss_component_usd_mwh"]
    - nyiso_price["congestion_component_usd_mwh"]
)

nyiso_price.head()

,timestamp_local,timestamp_utc,location_id,location,day_ahead_price_usd_mwh,loss_component_usd_mwh,congestion_component_usd_mwh,energy_component_usd_mwh
0,2025-01-01 00:00:00,2025-01-01 05:00:00+00:00,61758,HUD VL,33.16,1.31,0.0,31.85
1,2025-01-01 01:00:00,2025-01-01 06:00:00+00:00,61758,HUD VL,32.07,1.26,0.0,30.81
2,2025-01-01 02:00:00,2025-01-01 07:00:00+00:00,61758,HUD VL,30.02,1.16,0.0,28.86
3,2025-01-01 03:00:00,2025-01-01 08:00:00+00:00,61758,HUD VL,28.28,1.01,0.0,27.27
4,2025-01-01 04:00:00,2025-01-01 09:00:00+00:00,61758,HUD VL,28.22,1.03,0.0,27.19


In [196]:
print(
    nyiso_load_raw["Time Zone"].value_counts(
        dropna=False
    )
)

nyiso_load = nyiso_load_raw.copy()

nyiso_load["timestamp_local"] = pd.to_datetime(
    nyiso_load["Time Stamp"],
    format="%Y-%m-%d %H:%M:%S",
)

nyiso_load["timestamp_utc"] = (
    nyiso_load["timestamp_local"]
    .dt.tz_localize("America/New_York")
    .dt.tz_convert("UTC")
)

nyiso_load = nyiso_load[
    [
        "timestamp_local",
        "timestamp_utc",
        "Time Zone",
        "Name",
        "PTID",
        "Integrated Load",
    ]
].rename(
    columns={
        "Time Zone": "source_time_zone",
        "Name": "load_location",
        "PTID": "load_location_id",
        "Integrated Load": "actual_load_mw",
    }
)

nyiso_load = nyiso_load.sort_values(
    "timestamp_local"
).reset_index(drop=True)

Time Zone
EST    744
Name: count, dtype: int64


In [197]:
print("NYISO price shape:", nyiso_price.shape)
print("NYISO load shape:", nyiso_load.shape)

print(
    "NYISO price duplicate hours:",
    nyiso_price["timestamp_local"].duplicated().sum(),
)

print(
    "NYISO load duplicate hours:",
    nyiso_load["timestamp_local"].duplicated().sum(),
)

print(
    "NYISO missing prices:",
    nyiso_price["day_ahead_price_usd_mwh"].isna().sum(),
)

print(
    "NYISO missing load values:",
    nyiso_load["actual_load_mw"].isna().sum(),
)

assert len(nyiso_price) == 744
assert len(nyiso_load) == 744
assert nyiso_price["timestamp_local"].is_unique
assert nyiso_load["timestamp_local"].is_unique
assert nyiso_price["timestamp_utc"].is_unique
assert nyiso_load["timestamp_utc"].is_unique
assert nyiso_price["day_ahead_price_usd_mwh"].notna().all()
assert nyiso_load["actual_load_mw"].notna().all()

NYISO price shape: (744, 8)
NYISO load shape: (744, 6)
NYISO price duplicate hours: 0
NYISO load duplicate hours: 0
NYISO missing prices: 0
NYISO missing load values: 0


In [198]:
## Clean and validate weather data
JANUARY_HOURS = pd.date_range(
    start="2025-01-01 00:00:00",
    end="2025-01-31 23:00:00",
    freq="h",
)

In [199]:
def clean_weather(
    raw_weather: pd.DataFrame,
    station_code: str,
) -> pd.DataFrame:
    weather = raw_weather.copy()

    # Parse observation time.
    weather["observed_at"] = pd.to_datetime(
        weather["DATE"],
        errors="coerce",
    )

    # Keep January 2025 hourly/special weather reports.
    weather = weather[
        weather["observed_at"].between(
            "2025-01-01 00:00:00",
            "2025-01-31 23:59:59",
        )
        & weather["REPORT_TYPE"].isin(
            ["FM-12", "FM-15", "FM-16"]
        )
    ].copy()

    source_columns = {
        "HourlyDryBulbTemperature": "temperature_c",
        "HourlyDewPointTemperature": "dew_point_c",
        "HourlyRelativeHumidity": "relative_humidity_pct",
        "HourlyWindSpeed": "wind_speed_mps",
    }

    quality_flag_columns = []

    # Extract the numeric measurement while recording whether
    # NOAA attached a nonnumeric quality flag such as "s".
    for source_column, clean_column in source_columns.items():
        raw_values = weather[source_column].astype("string")

        flag_column = f"{clean_column}_quality_flagged"

        weather[flag_column] = (
            raw_values.notna()
            & raw_values.str.contains(
                r"[A-Za-z]",
                regex=True,
                na=False,
            )
        )

        weather[clean_column] = pd.to_numeric(
            raw_values.str.extract(
                r"([-+]?\d*\.?\d+)",
                expand=False,
            ),
            errors="coerce",
        )

        quality_flag_columns.append(flag_column)

    value_columns = [
        "temperature_c",
        "dew_point_c",
        "relative_humidity_pct",
        "wind_speed_mps",
    ]

    # Identify physically implausible observations.
    invalid_temperature = ~weather["temperature_c"].between(
        -50,
        50,
    )

    invalid_dew_point = ~weather["dew_point_c"].between(
        -60,
        40,
    )

    invalid_humidity = ~weather[
        "relative_humidity_pct"
    ].between(
        0,
        100,
    )

    invalid_wind_speed = ~weather[
        "wind_speed_mps"
    ].between(
        0,
        75,
    )

    weather["weather_value_rejected"] = (
        invalid_temperature
        | invalid_dew_point
        | invalid_humidity
        | invalid_wind_speed
    )

    # Reject the complete observation when any measurement
    # indicates that the record may be corrupted.
    weather.loc[
        weather["weather_value_rejected"],
        value_columns,
    ] = pd.NA

    weather["weather_quality_flagged"] = weather[
        quality_flag_columns
    ].any(axis=1)

    weather["timestamp_local"] = weather[
        "observed_at"
    ].dt.floor("h")

    # Prefer the routine aviation report. Use FM-12 or FM-16
    # only when an FM-15 report is unavailable for the hour.
    report_priority = {
        "FM-15": 1,
        "FM-12": 2,
        "FM-16": 3,
    }

    weather["report_priority"] = weather[
        "REPORT_TYPE"
    ].map(report_priority)

    hourly = (
        weather.sort_values(
            [
                "timestamp_local",
                "report_priority",
                "observed_at",
            ]
        )
        .drop_duplicates(
            subset="timestamp_local",
            keep="first",
        )
        .set_index("timestamp_local")
        .reindex(JANUARY_HOURS)
    )

    hourly["weather_missing"] = hourly[
        value_columns
    ].isna().any(axis=1)

    # No interpolation is performed here.
    hourly["weather_imputed"] = False
    hourly["weather_station"] = station_code
    hourly.index.name = "timestamp_local"

    output_columns = [
        "observed_at",
        "REPORT_TYPE",
        "temperature_c",
        "dew_point_c",
        "relative_humidity_pct",
        "wind_speed_mps",
        "weather_quality_flagged",
        "weather_value_rejected",
        "weather_missing",
        "weather_imputed",
        "weather_station",
    ]

    return hourly[output_columns].reset_index()

In [200]:
newark_weather = clean_weather(
    newark_weather_raw,
    station_code="USW00014734",
)

stewart_weather = clean_weather(
    stewart_weather_raw,
    station_code="USW00014714",
)

In [201]:
weather_value_columns = [
    "temperature_c",
    "dew_point_c",
    "relative_humidity_pct",
    "wind_speed_mps",
]


def summarize_weather(
    weather: pd.DataFrame,
    station_name: str,
) -> None:
    print(f"\n{station_name}")
    print("Rows:", len(weather))
    print(
        "Duplicate hours:",
        weather["timestamp_local"].duplicated().sum(),
    )
    print(
        "Missing hours or values:",
        weather["weather_missing"].sum(),
    )
    print(
        "Quality-flagged observations:",
        weather["weather_quality_flagged"].fillna(False).sum(),
    )
    print(
        "Rejected observations:",
        weather["weather_value_rejected"].fillna(False).sum(),
    )
    print("\nMissing values by column:")
    print(weather[weather_value_columns].isna().sum())

    assert len(weather) == 744
    assert weather["timestamp_local"].notna().all()
    assert weather["timestamp_local"].is_unique
    assert weather["timestamp_local"].is_monotonic_increasing


summarize_weather(newark_weather, "Newark")
summarize_weather(stewart_weather, "Stewart")


Newark
Rows: 744
Duplicate hours: 0
Missing hours or values: 3
Quality-flagged observations: 0
Rejected observations: 0

Missing values by column:
temperature_c            3
dew_point_c              3
relative_humidity_pct    3
wind_speed_mps           3
dtype: int64

Stewart
Rows: 744
Duplicate hours: 0
Missing hours or values: 7
Quality-flagged observations: 7
Rejected observations: 4

Missing values by column:
temperature_c            7
dew_point_c              7
relative_humidity_pct    7
wind_speed_mps           7
dtype: int64


In [202]:
datasets = {
    "PJM price": pjm_price_raw,
    "PJM load": pjm_load_raw,
    "NYISO price": nyiso_price_raw,
    "NYISO load": nyiso_load_raw,
    "Newark weather": newark_weather_raw,
    "Stewart weather": stewart_weather_raw,
}

for name, df in datasets.items():
    print(f"{name:20} rows={df.shape[0]:5} columns={df.shape[1]:3}")

PJM price            rows=  744 columns= 14
PJM load             rows=  744 columns=  8
NYISO price          rows=  744 columns=  7
NYISO load           rows=  744 columns=  6
Newark weather       rows=12970 columns=125
Stewart weather      rows= 9081 columns=125


In [203]:
assert pjm_price_raw.shape == (744, 14)
assert pjm_load_raw.shape == (744, 8)
assert nyiso_price_raw.shape == (744, 7)
assert nyiso_load_raw.shape == (744, 6)

print("All four electricity files passed the row-count checks.")

All four electricity files passed the row-count checks.


In [204]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.columns.tolist())


PJM price
['datetime_beginning_utc', 'datetime_beginning_ept', 'pnode_id', 'pnode_name', 'voltage', 'equipment', 'type', 'zone', 'system_energy_price_da', 'total_lmp_da', 'congestion_price_da', 'marginal_loss_price_da', 'row_is_current', 'version_nbr']

PJM load
['datetime_beginning_utc', 'datetime_beginning_ept', 'nerc_region', 'mkt_region', 'zone', 'load_area', 'mw', 'is_verified']

NYISO price
['Time Stamp', 'Name', 'PTID', 'LBMP ($/MWHr)', 'Marginal Cost Losses ($/MWHr)', 'Marginal Cost Congestion ($/MWHr)', 'source_file']

NYISO load
['Time Stamp', 'Time Zone', 'Name', 'PTID', 'Integrated Load', 'source_file']

Newark weather
['STATION', 'DATE', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'REPORT_TYPE', 'SOURCE', 'HourlyAltimeterSetting', 'HourlyDewPointTemperature', 'HourlyDryBulbTemperature', 'HourlyPrecipitation', 'HourlyPresentWeatherType', 'HourlyPressureChange', 'HourlyPressureTendency', 'HourlyRelativeHumidity', 'HourlySkyConditions', 'HourlySeaLevelPressure', 'HourlySta

In [205]:
def validate_hourly_table(
    df: pd.DataFrame,
    table_name: str,
    value_column: str,
) -> None:
    expected_rows = 744

    assert len(df) == expected_rows, (
        f"{table_name}: expected {expected_rows} rows, "
        f"but found {len(df)}"
    )

    assert df["timestamp_local"].notna().all(), (
        f"{table_name}: missing local timestamps"
    )

    assert df["timestamp_utc"].notna().all(), (
        f"{table_name}: missing UTC timestamps"
    )

    assert df["timestamp_local"].is_unique, (
        f"{table_name}: duplicate local timestamps"
    )

    assert df["timestamp_utc"].is_unique, (
        f"{table_name}: duplicate UTC timestamps"
    )

    assert df["timestamp_local"].is_monotonic_increasing, (
        f"{table_name}: local timestamps are not sorted"
    )

    assert df[value_column].notna().all(), (
        f"{table_name}: missing values in {value_column}"
    )

    print(f"{table_name} passed validation.")

In [206]:
validate_hourly_table(
    pjm_price,
    "PJM price",
    "day_ahead_price_usd_mwh",
)

validate_hourly_table(
    pjm_load,
    "PJM load",
    "actual_load_mw",
)

validate_hourly_table(
    nyiso_price,
    "NYISO price",
    "day_ahead_price_usd_mwh",
)

validate_hourly_table(
    nyiso_load,
    "NYISO load",
    "actual_load_mw",
)

PJM price passed validation.
PJM load passed validation.
NYISO price passed validation.
NYISO load passed validation.


In [207]:
pjm_electricity = pjm_price.merge(
    pjm_load,
    on=["timestamp_local", "timestamp_utc"],
    how="outer",
    validate="one_to_one",
    indicator=True,
)

print(pjm_electricity["_merge"].value_counts())
print("PJM merged rows:", len(pjm_electricity))

assert len(pjm_electricity) == 744
assert pjm_electricity["_merge"].eq("both").all()

pjm_electricity = pjm_electricity.drop(columns="_merge")

pjm_electricity.head()

_merge
both          744
left_only       0
right_only      0
Name: count, dtype: int64
PJM merged rows: 744


,timestamp_local,timestamp_utc,location_id,location,day_ahead_price_usd_mwh,energy_component_usd_mwh,congestion_component_usd_mwh,loss_component_usd_mwh,zone,load_area,actual_load_mw,is_verified
0,2025-01-01 00:00:00,2025-01-01 05:00:00+00:00,51301,PSEG,21.981200,21.26,0.153437,0.567763,PS,PS,4068.850,True
1,2025-01-01 01:00:00,2025-01-01 06:00:00+00:00,51301,PSEG,21.397873,20.96,-0.012770,0.450643,PS,PS,3966.606,True
2,2025-01-01 02:00:00,2025-01-01 07:00:00+00:00,51301,PSEG,20.641678,20.42,-0.266470,0.488148,PS,PS,3851.354,True
3,2025-01-01 03:00:00,2025-01-01 08:00:00+00:00,51301,PSEG,20.696320,20.45,-0.283920,0.530240,PS,PS,3767.111,True
4,2025-01-01 04:00:00,2025-01-01 09:00:00+00:00,51301,PSEG,20.649481,20.46,-0.400733,0.590214,PS,PS,3766.670,True


In [208]:
nyiso_electricity = nyiso_price.merge(
    nyiso_load,
    on=["timestamp_local", "timestamp_utc"],
    how="outer",
    validate="one_to_one",
    indicator=True,
)

print(nyiso_electricity["_merge"].value_counts())
print("NYISO merged rows:", len(nyiso_electricity))

assert len(nyiso_electricity) == 744
assert nyiso_electricity["_merge"].eq("both").all()

nyiso_electricity = nyiso_electricity.drop(columns="_merge")

nyiso_electricity.head()

_merge
both          744
left_only       0
right_only      0
Name: count, dtype: int64
NYISO merged rows: 744


,timestamp_local,timestamp_utc,location_id,location,day_ahead_price_usd_mwh,loss_component_usd_mwh,congestion_component_usd_mwh,energy_component_usd_mwh,source_time_zone,load_location,load_location_id,actual_load_mw
0,2025-01-01 00:00:00,2025-01-01 05:00:00+00:00,61758,HUD VL,33.16,1.31,0.0,31.85,EST,HUD VL,61758,958.0416
1,2025-01-01 01:00:00,2025-01-01 06:00:00+00:00,61758,HUD VL,32.07,1.26,0.0,30.81,EST,HUD VL,61758,915.9221
2,2025-01-01 02:00:00,2025-01-01 07:00:00+00:00,61758,HUD VL,30.02,1.16,0.0,28.86,EST,HUD VL,61758,884.2183
3,2025-01-01 03:00:00,2025-01-01 08:00:00+00:00,61758,HUD VL,28.28,1.01,0.0,27.27,EST,HUD VL,61758,864.1147
4,2025-01-01 04:00:00,2025-01-01 09:00:00+00:00,61758,HUD VL,28.22,1.03,0.0,27.19,EST,HUD VL,61758,860.9837


In [209]:
# Export processed electricity datasets

processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

pjm_output_path = (
    processed_dir
    / "pjm_pseg_january_2025_electricity.csv"
)

nyiso_output_path = (
    processed_dir
    / "nyiso_hudson_valley_january_2025_electricity.csv"
)

pjm_electricity.to_csv(
    pjm_output_path,
    index=False,
)

nyiso_electricity.to_csv(
    nyiso_output_path,
    index=False,
)

print("Saved:", pjm_output_path.resolve())
print("Saved:", nyiso_output_path.resolve())

Saved: C:\Users\david\Desktop\Data Science\DATA 698\electricity-price-forecasting\data\processed\pjm_pseg_january_2025_electricity.csv
Saved: C:\Users\david\Desktop\Data Science\DATA 698\electricity-price-forecasting\data\processed\nyiso_hudson_valley_january_2025_electricity.csv


In [210]:
assert pjm_output_path.exists()
assert nyiso_output_path.exists()

print("PJM exported rows:", len(pd.read_csv(pjm_output_path)))
print("NYISO exported rows:", len(pd.read_csv(nyiso_output_path)))

PJM exported rows: 744
NYISO exported rows: 744
